# Task 2: Hands-On Data Lab

Analyze a small fictional employee dataset. Run all cells from top to bottom to validate the records, inspect summary metrics, create four charts, and export the results and Word report. The sample is illustrative only.

## 1. Import libraries and configure project paths

In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt, RGBColor

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'dataset.csv').exists():
    PROJECT_DIR = PROJECT_DIR / 'Task-2-Hands-On-Data-Lab'
DATA_PATH = PROJECT_DIR / 'dataset.csv'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
CHART_DIR = OUTPUT_DIR / 'charts'
REPORT_PATH = PROJECT_DIR / 'report' / 'Task_2_Hands_On_Data_Lab_Report.docx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Project folder: {PROJECT_DIR}')

## 2. Load and validate the dataset

In [ ]:
required_columns = ['Employee_ID', 'Department', 'Experience_Years', 'Salary', 'Sales_Target_Met', 'Performance_Score']
data = pd.read_csv(DATA_PATH)
missing_columns = set(required_columns).difference(data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if len(data) != 15:
    raise ValueError(f'Expected 15 records; found {len(data)}')
if data[required_columns].isna().any().any():
    raise ValueError('Required fields contain missing values')
if data['Employee_ID'].duplicated().any():
    raise ValueError('Employee_ID values must be unique')
if not set(data['Department']).issubset({'Sales', 'Marketing', 'Engineering', 'HR'}):
    raise ValueError('Unexpected department category')
if not set(data['Sales_Target_Met']).issubset({'Yes', 'No'}):
    raise ValueError('Sales_Target_Met must be Yes or No')
for column in ['Experience_Years', 'Salary', 'Performance_Score']:
    if not pd.api.types.is_numeric_dtype(data[column]):
        raise TypeError(f'{column} must be numeric')
if (data['Experience_Years'] < 0).any() or (data['Salary'] <= 0).any():
    raise ValueError('Experience must be non-negative and salary must be positive')
if not data['Performance_Score'].between(0, 5).all():
    raise ValueError('Performance_Score must be between 0 and 5')
print(f'Rows: {data.shape[0]} | Columns: {data.shape[1]}')
display(data.head())

## 3. Exploratory data analysis and data-quality checks

In [ ]:
print('Data types:')
display(data.dtypes.to_frame('dtype'))
print('Missing values by column:')
display(data.isna().sum().to_frame('missing_count'))
print(f'Duplicate employee IDs: {data.duplicated(subset=["Employee_ID"]).sum()}')
print('Department counts:')
display(data['Department'].value_counts().rename_axis('Department').to_frame('Employee_Count'))
print('Sales target status counts:')
display(data['Sales_Target_Met'].value_counts().rename_axis('Sales_Target_Met').to_frame('Employee_Count'))

## 4. Numeric and department summary statistics

In [ ]:
numeric_columns = ['Experience_Years', 'Salary', 'Performance_Score']
numeric_summary = data[numeric_columns].describe()
department_summary = data.groupby('Department').agg(
    Employee_Count=('Employee_ID', 'count'),
    Average_Experience_Years=('Experience_Years', 'mean'),
    Average_Salary=('Salary', 'mean'),
    Median_Salary=('Salary', 'median'),
    Average_Performance_Score=('Performance_Score', 'mean'),
).sort_values('Average_Salary', ascending=False)
print('Descriptive statistics:')
display(numeric_summary)
print('Department aggregations:')
display(department_summary)
print(f'NumPy mean salary: ${np.mean(data["Salary"].to_numpy()):,.2f}')
print(f'Experience-salary correlation: {np.corrcoef(data["Experience_Years"], data["Salary"])[0, 1]:.3f}')

## 5. Create four high-resolution charts

In [ ]:
sns.set_theme(style='whitegrid', palette='colorblind')
chart_paths = []
def save_chart(filename):
    path = CHART_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    chart_paths.append(path)

plt.figure(figsize=(8, 5))
sns.histplot(data=data, x='Salary', bins=7, kde=True, color='#26766f')
plt.title('Employee Salary Distribution')
plt.xlabel('Annual salary (USD)')
plt.ylabel('Number of employees')
save_chart('histogram.png')

plt.figure(figsize=(8, 5))
sns.barplot(data=data, x='Department', y='Salary', errorbar=None, color='#4c78a8')
plt.title('Average Salary by Department')
plt.xlabel('Department')
plt.ylabel('Average annual salary (USD)')
plt.ticklabel_format(axis='y', style='plain')
save_chart('bar_chart.png')

plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x='Experience_Years', y='Salary', hue='Department', style='Department', s=85)
plt.title('Experience and Annual Salary')
plt.xlabel('Experience (years)')
plt.ylabel('Annual salary (USD)')
plt.ticklabel_format(axis='y', style='plain')
save_chart('scatter_plot.png')

plt.figure(figsize=(7, 5))
sns.boxplot(data=data, x='Sales_Target_Met', y='Performance_Score', order=['No', 'Yes'])
sns.stripplot(data=data, x='Sales_Target_Met', y='Performance_Score', order=['No', 'Yes'], color='#333333', alpha=0.55, jitter=0.12, size=4)
plt.title('Performance Score by Sales Target Status')
plt.xlabel('Sales target met')
plt.ylabel('Performance score (0-5)')
save_chart('box_plot.png')
print('Saved charts:', [path.name for path in chart_paths])

## 6. Export results and build the Word report

In [ ]:
results_lines = [
    'TASK 2 - HANDS-ON DATA LAB RESULTS', '=' * 38,
    f'Records: {len(data)}', f'Columns: {len(data.columns)}',
    f'Duplicate employee IDs: {int(data["Employee_ID"].duplicated().sum())}', '',
    'Missing values by column:', data.isna().sum().to_string(), '',
    'Numeric descriptive statistics:', numeric_summary.to_string(float_format=lambda value: f'{value:,.2f}'), '',
    'Department aggregations:', department_summary.to_string(float_format=lambda value: f'{value:,.2f}'), '',
    'Sales target status counts:', data['Sales_Target_Met'].value_counts().sort_index().to_string(), '',
    'Interpretation note: This small, fictional sample is illustrative and is not representative of a real workforce.'
]
(OUTPUT_DIR / 'results.txt').write_text('\n'.join(results_lines) + '\n', encoding='utf-8')

doc = Document()
doc.styles['Normal'].font.name = 'Aptos'
doc.styles['Normal'].font.size = Pt(10)
doc.styles['Title'].font.name = 'Aptos Display'
doc.styles['Title'].font.color.rgb = RGBColor(32, 91, 94)
title = doc.add_heading('Task 2: Hands-On Data Lab', level=0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
subtitle = doc.add_paragraph('Employee compensation and performance analysis')
subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
target_rate = data['Sales_Target_Met'].eq('Yes').mean() * 100
correlation = data['Experience_Years'].corr(data['Salary'])
doc.add_heading('Executive Summary', level=1)
doc.add_paragraph(
    f'This analysis describes {len(data)} fictional employee records across '
    f'{data["Department"].nunique()} departments. Average salary is '
    f'${data["Salary"].mean():,.0f}; median salary is ${data["Salary"].median():,.0f}; '
    f'average performance is {data["Performance_Score"].mean():.2f} / 5. '
    f'{target_rate:.1f}% met the sales target. Experience-salary correlation is '
    f'{correlation:.2f}; this descriptive association does not establish causation.'
)
doc.add_heading('Analytical Methodology', level=1)
doc.add_paragraph(
    'The CSV was loaded with Pandas and checked for schema, missing values, unique IDs, '
    'allowed categories, and plausible numeric values. NumPy and Pandas produced descriptive '
    'statistics and department aggregations. Seaborn and Matplotlib generated four 300-DPI '
    'charts. Findings are descriptive and limited by the small, fictional sample.'
)
doc.add_heading('Summary Metrics', level=1)
metrics = [
    ('Employee records', str(len(data))),
    ('Departments', str(data['Department'].nunique())),
    ('Average salary', f'${data["Salary"].mean():,.2f}'),
    ('Median salary', f'${data["Salary"].median():,.2f}'),
    ('Average experience', f'{data["Experience_Years"].mean():.2f} years'),
    ('Average performance', f'{data["Performance_Score"].mean():.2f} / 5'),
    ('Sales target met', f'{target_rate:.1f}%'),
    ('Experience-salary correlation', f'{correlation:.2f}'),
    ('Missing values', str(int(data.isna().sum().sum())))
]
table = doc.add_table(rows=1, cols=2)
table.style = 'Light Shading Accent 1'
table.rows[0].cells[0].text = 'Metric'
table.rows[0].cells[1].text = 'Value'
for metric, value in metrics:
    cells = table.add_row().cells
    cells[0].text = metric
    cells[1].text = value
doc.add_paragraph(
    'Department average salaries: ' + '; '.join(
        f'{department}: ${row["Average_Salary"]:,.0f}'
        for department, row in department_summary.iterrows()
    ) + '.'
)
doc.add_heading('Visual Analysis', level=1)
chart_titles = {
    'histogram.png': 'Salary Distribution',
    'bar_chart.png': 'Average Salary by Department',
    'scatter_plot.png': 'Experience and Salary',
    'box_plot.png': 'Performance by Sales Target Status'
}
for chart_path in chart_paths:
    doc.add_heading(chart_titles[chart_path.name], level=2)
    paragraph = doc.add_paragraph()
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.add_run().add_picture(str(chart_path), width=Inches(6.0))
doc.add_paragraph('Source: included fictional teaching dataset. Results are descriptive only.')
doc.save(REPORT_PATH)
print(f'Results saved to {OUTPUT_DIR / "results.txt"}')
print(f'Word report saved to {REPORT_PATH}')